# Advanced Analytics + Risk Metrics
Bluestock Mutual Fund Analytics Capstone

This notebook covers: Historical VaR/CVaR, rolling 90-day Sharpe ratio, investor cohort analysis,
SIP continuity analysis, a simple fund recommender, and sector concentration (HHI).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../bluestock_mf.db")
plt.style.use("seaborn-v0_8-whitegrid")


## 1. Historical VaR (95%) and CVaR

- **VaR (95%)** = 5th percentile of the daily return distribution (the loss level not expected to be exceeded 95% of the time).
- **CVaR** (Conditional VaR / Expected Shortfall) = mean of returns that fall below the VaR threshold — i.e. the average loss *in the worst 5% of days*.

Computed for all 40 schemes.


In [ ]:
nav = pd.read_sql("SELECT amfi_code, date, nav_value FROM fact_nav ORDER BY amfi_code, date", engine)
nav["date"] = pd.to_datetime(nav["date"])

fund_names = pd.read_sql("SELECT amfi_code, scheme_name FROM dim_fund", engine)

var_cvar_rows = []
for code_, grp in nav.groupby("amfi_code"):
    grp = grp.sort_values("date")
    returns = grp["nav_value"].pct_change().dropna()
    if len(returns) < 30:
        continue
    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()
    var_cvar_rows.append({"amfi_code": code_, "var_95_pct": var_95 * 100, "cvar_95_pct": cvar_95 * 100})

var_cvar_df = pd.DataFrame(var_cvar_rows).merge(fund_names, on="amfi_code")
var_cvar_df = var_cvar_df[["amfi_code", "scheme_name", "var_95_pct", "cvar_95_pct"]].sort_values("var_95_pct")

var_cvar_df.to_csv("../reports/var_cvar_report.csv", index=False)
print(f"Saved var_cvar_report.csv — {len(var_cvar_df)} schemes")
var_cvar_df.head(10)


In [ ]:
# 5 funds with the worst (most negative) VaR - i.e. highest downside risk on a bad day
var_cvar_df.sort_values("var_95_pct").head(5)


## 2. Rolling 90-Day Sharpe Ratio

`rolling_sharpe = returns.rolling(90).mean() / returns.rolling(90).std() * sqrt(252)`

Plotted over time for 5 key funds (the top 5 from the Day 4 Fund Scorecard).


In [ ]:
top5_names = [
    "Mirae Asset Large Cap Fund - Regular - Growth",
    "ICICI Pru Midcap Fund - Regular - Growth",
    "Kotak Flexicap Fund - Regular - Growth",
    "HDFC Mid-Cap Opportunities Fund - Regular - Growth",
    "ICICI Pru Bluechip Fund - Direct - Growth",
]
top5_codes = fund_names[fund_names["scheme_name"].isin(top5_names)]

fig, ax = plt.subplots(figsize=(12, 6))
for _, row in top5_codes.iterrows():
    fund_nav = nav[nav["amfi_code"] == row["amfi_code"]].sort_values("date").copy()
    fund_nav["returns"] = fund_nav["nav_value"].pct_change()
    fund_nav["rolling_sharpe"] = (
        fund_nav["returns"].rolling(90).mean() / fund_nav["returns"].rolling(90).std() * np.sqrt(252)
    )
    ax.plot(fund_nav["date"], fund_nav["rolling_sharpe"], label=row["scheme_name"], linewidth=1.3)

ax.set_title("Rolling 90-Day Sharpe Ratio — Top 5 Scorecard Funds")
ax.set_xlabel("Date")
ax.set_ylabel("Rolling Sharpe Ratio (annualized)")
ax.legend(fontsize=8, loc="upper left")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.savefig("../reports/charts/rolling_sharpe_chart.png", dpi=150)
plt.show()
print("Saved rolling_sharpe_chart.png")


## 3. Investor Cohort Analysis

Group investors by the year of their **first transaction**. For each cohort, compute:
- Average SIP amount
- Total invested
- Top fund preference


In [ ]:
txn = pd.read_sql("SELECT * FROM fact_transactions", engine)
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"])

first_txn = txn.groupby("investor_id")["transaction_date"].min().reset_index()
first_txn["cohort_year"] = first_txn["transaction_date"].dt.year
txn_cohort = txn.merge(first_txn[["investor_id", "cohort_year"]], on="investor_id")

sip_only = txn_cohort[txn_cohort["transaction_type"] == "Sip"]
cohort_summary = sip_only.groupby("cohort_year").agg(
    avg_sip_amount=("amount_inr", "mean"),
    total_invested=("amount_inr", "sum"),
    num_investors=("investor_id", "nunique"),
).reset_index()

# top fund preference per cohort
fund_counts = txn_cohort.groupby(["cohort_year", "amfi_code"]).size().reset_index(name="txn_count")
top_fund_per_cohort = (
    fund_counts.sort_values(["cohort_year", "txn_count"], ascending=[True, False])
    .groupby("cohort_year", as_index=False).first()
    .merge(fund_names, on="amfi_code")[["cohort_year", "scheme_name"]]
    .rename(columns={"scheme_name": "top_fund"})
)

cohort_summary = cohort_summary.merge(top_fund_per_cohort, on="cohort_year")
cohort_summary


## 4. SIP Continuity Analysis

For investors with 6+ SIP transactions, compute the average gap (in days) between consecutive SIP dates.
Investors with an average gap > 35 days are flagged **"at-risk"** (missing the expected ~monthly cadence).


In [ ]:
sip_txns = txn[txn["transaction_type"] == "Sip"].sort_values(["investor_id", "transaction_date"])
sip_counts = sip_txns.groupby("investor_id").size()
eligible_investors = sip_counts[sip_counts >= 6].index

continuity_rows = []
for inv_id, grp in sip_txns[sip_txns["investor_id"].isin(eligible_investors)].groupby("investor_id"):
    dates = grp["transaction_date"].sort_values()
    gaps = dates.diff().dropna().dt.days
    avg_gap = gaps.mean()
    continuity_rows.append({"investor_id": inv_id, "num_sips": len(grp), "avg_gap_days": avg_gap})

continuity_df = pd.DataFrame(continuity_rows)
continuity_df["at_risk"] = continuity_df["avg_gap_days"] > 35

at_risk_pct = continuity_df["at_risk"].mean() * 100
print(f"Investors with 6+ SIPs: {len(continuity_df)}")
print(f"At-risk (avg gap > 35 days): {continuity_df['at_risk'].sum()} ({at_risk_pct:.1f}%)")
continuity_df.sort_values("avg_gap_days", ascending=False).head(10)


## 5. Simple Fund Recommender

Input: risk appetite (Low / Moderate / High / Very High).
Output: top 3 funds by Sharpe ratio within the matching `risk_grade`.

(This logic also lives standalone in `Scripts/recommender.py` for interactive command-line use.)


In [ ]:
def recommend(risk_appetite):
    perf = pd.read_sql("""
        SELECT dim_fund.scheme_name, dim_fund.fund_house, fact_performance.sharpe_ratio,
               fact_performance.risk_grade, fact_performance.return_3yr_pct
        FROM fact_performance
        JOIN dim_fund USING (amfi_code)
        WHERE fact_performance.risk_grade = ?
        ORDER BY fact_performance.sharpe_ratio DESC
        LIMIT 3
    """, engine, params=(risk_appetite,))
    return perf

for appetite in ["Low", "Moderate", "High", "Very High"]:
    print(f"--- {appetite} ---")
    print(recommend(appetite).to_string(index=False))
    print()


## 6. Sector Concentration — Herfindahl-Hirschman Index (HHI)

`HHI = sum(weight_i^2)` per fund, using portfolio holdings weights (as fractions, not percentages).
Higher HHI = more concentrated in fewer sectors. Compared across all equity-category funds.


In [ ]:
holdings = pd.read_csv("../Data/Processed/09_portfolio_holdings.csv")

sector_weights = holdings.groupby(["amfi_code", "sector"])["weight_pct"].sum().reset_index()
sector_weights["weight_frac"] = sector_weights["weight_pct"] / 100

hhi_df = sector_weights.groupby("amfi_code").apply(
    lambda g: (g["weight_frac"] ** 2).sum()
).reset_index(name="hhi")

equity_funds = pd.read_sql("SELECT amfi_code, scheme_name, category FROM dim_fund WHERE category = 'Equity'", engine)
hhi_df = hhi_df.merge(equity_funds, on="amfi_code").sort_values("hhi", ascending=False)

print(f"HHI computed for {len(hhi_df)} equity funds")
hhi_df[["scheme_name", "category", "hhi"]].head(10)


## 7. Key Advanced Insights

1. **Highest VaR risk**: The schemes with the most negative 95% VaR (see the sorted table in Section 1) represent the funds most exposed to single-day downside — typically Small Cap and Sectoral/Thematic categories, consistent with their "Very High" risk grades from the Day 4 Fund Scorecard.

2. **Rolling Sharpe volatility**: The 90-day rolling Sharpe ratio for the top 5 scorecard funds fluctuates meaningfully over time rather than staying flat — reinforcing that the static, full-period Sharpe ratio used in the Fund Scorecard is a long-run average, not a guarantee of consistent risk-adjusted performance in any given quarter.

3. **Investor cohorts**: Later-joining cohorts show different average SIP amounts and fund preferences compared to earlier cohorts, reflecting how investor behavior and market conditions shifted over the dataset's 2022–2026 span (see Section 3 table for exact cohort-by-cohort figures).

4. **SIP continuity / at-risk investors**: A measurable share of investors with 6+ SIP transactions have an average gap exceeding 35 days between contributions — these are flagged as "at-risk" of SIP discontinuation and could be a useful segment for Bluestock's retention outreach.

5. **Sector concentration (HHI)**: Equity funds show a range of HHI values — funds concentrated in a few sectors (higher HHI) carry more sector-specific risk than well-diversified funds (lower HHI), which is a useful complement to the fund-level risk grade when advising more risk-conscious investors.
